This notebook focuses on parsing macOS system logs into structured events. The goal is not analysis or detection, but reliable extraction of consistent fields from unstructured, multi-line log entries.

Jan 23 00:30:21 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.cdscheduler" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.install" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.

Above we can see the raw log and example of logs notices/alerts that have come through. So, we need to break it down. Timestamp, device, service, process ID (PID), message.
We need to consider things like; when does a new log start, what is the delimiter for the message, how can we determine the split between each section of the log etc.

An important first step in breaking down the logs is to extract each log. Where does it start and end? This is easy as we know each log starts with a timestamp. Therefore, multi-line log messages don't matter, these are all included as the end of the message is when the next log starts i.e. where the next timestamp is.

Let's get into the code:

In [2]:
from pathlib import Path
import re
from datetime import datetime
import pandas as pd

In [3]:
log_path = Path("../datasets/raw/system.log")

with log_path.open("r") as f:
    lines = f.readlines()

len(lines)

84

In [4]:
for line in lines[:10]:
    print(line.rstrip())

Jan 23 00:14:01 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:21 Peters-Air syslogd[388]: ASL Sender Statistics
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.cdscheduler" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.install" claims selected messages.
	Those messages may not appear in standard system log files or in the ASL database.
Jan 23 00:30:22 Peters-Air syslogd[388]: Configuration Notice:
	ASL Module "com.apple.authd" sharing output destination "/var/log/asl" with ASL Module "com.apple.asl".


In [5]:
header_pattern = re.compile(
    r"^(?P<month>\w{3})\s+(?P<day>\d{1,2})\s+(?P<hour>\d{2}):(?P<minute>\d{2}):(?P<second>\d{2})\s+(?P<host>\S+)\s+(?P<process>\S+)(?:\[(?P<pid>\d+)\])?:\s+(?P<message>.+)$"
)

In [ ]:
events = []
current_event = None
current_message = []

for line in lines:
    match = header_pattern.match(line)

    if match:
        if current_event:
            current_event["message"] = " ".join(current_message).strip()
            events.append(current_event)

        current_event = match.groupdict()
        message_part = line[match.end():].strip()
        current_message = [message_part] if message_part else []

    else:
        if current_event:
            current_message.append(line.strip())

if current_event:
    current_event["message"] = " ".join(current_message).strip()
    events.append(current_event)   

In [ ]:
len(events)
events[0]

In [7]:
df = pd.DataFrame(events)
df.head()

,month,day,hour,minute,second,host,process,pid,message
0,Jan,23,00,14,01,Peters-Air,syslogd[388],None,
1,Jan,23,00,30,21,Peters-Air,syslogd[388],None,
2,Jan,23,00,30,22,Peters-Air,syslogd[388],None,"ASL Module ""com.apple.cdscheduler"" claims sele..."
3,Jan,23,00,30,22,Peters-Air,syslogd[388],None,"ASL Module ""com.apple.install"" claims selected..."
4,Jan,23,00,30,22,Peters-Air,syslogd[388],None,"ASL Module ""com.apple.authd"" sharing output de..."
